In [1]:
import sys
import os
package_path = os.path.abspath("..")
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from dask.distributed import Client, LocalCluster

In [4]:
cluster=LocalCluster()
client=Client(cluster)

In [5]:
#load some data, filter, and 
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/shendure/shendure_counts_grouped.txt")
filtered,dropped_groups=scm.filter_low_umi_count(dat)
dropped_groups

{'cre_id': ['Col1a1_chr11_15306', 'Col1a2_chr6_65', 'minP_w_20bp_buffer'],
 'cell_type': []}

In [6]:
#run first time & cache on disc. If it's not the first time, load from disc instead. 
first_run=False

primordial=None

if first_run:
    primordial=scm.ortho()
    primordial.criss_cross(client=client,dat=filtered)
    primordial.extract_params(client)
    primordial.save('/gpfs/gibbs/pi/reilly/tabula_data/shendure','ortho')
else:
    primordial=scm.ortho.load(client,'/gpfs/gibbs/pi/reilly/tabula_data/shendure','ortho')

In [7]:
#create a simulation batch object
batch=scm.simulation_batch(primordial)
batch.describe_primordial(client,filtered)

In [8]:
description=batch.description_primordial_by_cre.compute()
description.to_csv("shendure_prim.tsv",sep="\t")

In [13]:
description.sort_values(by="nb")
#max(description["nb"])

,C(cell_type)[Cardiomyocytes],C(cell_type)[EpiblastPrimitiveStreak],C(cell_type)[ExEndodermParietal],C(cell_type)[ExEndodermVisceral],C(cell_type)[Haematoendothelial],C(cell_type)[Mesoderm],C(cell_type)[NeuroectodermBrain],C(cell_type)[NeuroectodermRostral],C(cell_type)[Pluripotent],C(cell_type)[SurfaceEctoderm],...,C(rep_id)[B1],C(rep_id)[B2],cre_id,cells,nb,zi,theta,r,sigmasquare,p
4291,0,0,0,1,0,0,0,0,0,0,...,0,0,Sox17_chr1_83,22,7.090039e-262,0.755683,-0.917572,0.399488,7.090039e-262,1.000000e+00
4284,0,0,0,1,0,0,0,0,0,0,...,0,1,Sox17_chr1_83,33,7.090039e-262,0.805551,-0.917572,0.399488,7.090039e-262,1.000000e+00
4281,0,0,0,1,0,0,0,0,0,0,...,0,0,Sox17_chr1_83,36,7.090039e-262,0.506100,-0.917572,0.399488,7.090039e-262,1.000000e+00
4278,0,0,0,1,0,0,0,0,0,0,...,1,0,Sox17_chr1_83,49,7.090039e-262,0.805446,-0.917572,0.399488,7.090039e-262,1.000000e+00
4269,0,0,0,1,0,0,0,0,0,0,...,0,0,Sox17_chr1_83,61,7.090039e-262,0.929513,-0.917572,0.399488,7.090039e-262,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3388,0,1,0,0,0,0,0,0,0,0,...,0,1,eef1aP,163,1.451269e+138,0.000498,-4.877599,0.007615,2.765732e+278,5.247323e-141
3386,0,1,0,0,0,0,0,0,0,0,...,0,0,eef1aP,213,1.451269e+138,0.000618,-4.877599,0.007615,2.765732e+278,5.247323e-141
3385,0,1,0,0,0,0,0,0,0,0,...,0,0,eef1aP,220,1.451269e+138,0.000608,-4.877599,0.007615,2.765732e+278,5.247323e-141
3375,0,1,0,0,0,0,0,0,0,0,...,0,0,eef1aP,292,1.451269e+138,0.000251,-4.877599,0.007615,2.765732e+278,5.247323e-141


In [26]:
description[description["cre_id"]=="eef1aP"][description["C(rep_id)[B2]"]==1][description["C(cell_type)[Pluripotent]"]==1]

,C(cell_type)[Cardiomyocytes],C(cell_type)[EpiblastPrimitiveStreak],C(cell_type)[ExEndodermParietal],C(cell_type)[ExEndodermVisceral],C(cell_type)[Haematoendothelial],C(cell_type)[Mesoderm],C(cell_type)[NeuroectodermBrain],C(cell_type)[NeuroectodermRostral],C(cell_type)[Pluripotent],C(cell_type)[SurfaceEctoderm],...,C(rep_id)[B1],C(rep_id)[B2],cre_id,cells,nb,zi,theta,r,sigmasquare,p
3358,0,0,0,0,0,0,0,0,1,0,...,0,1,eef1aP,508,1.662389e+106,0.000498,-4.877599,0.007615,3.628938e+214,4.580923e-109


In [32]:
set_of_interest=filtered[filtered["cre_id"]=="eef1aP"][filtered["rep_id"]=="B2"][filtered["cell_type"]=="Pluripotent"]

In [35]:
set_of_interest["umis_mpra_bc"].agg('mean')

375.93110236220474

This is a problem : the actual mean is like `376`, but our estimation is `1.662389e+106`. Where is the error coming from? Let's back up a few steps. Do the betas look OK?

In [15]:
r,p=scm.simulate_from_description(batch.description_primordial_by_cre)

scMPRAforge: WARNING: [simulate_from_description] explode took 0.21 seconds


In [10]:
import numpy as np

In [11]:
len(r)

778015

In [ ]:
interval=100000
for i in [1,2,3,4]:
    print((interval*(i-1),interval*i))
    toy_r=r[interval*(i-1):interval*i]
    toy_p=p[interval*(i-1):interval*i]
    np.random.negative_binomial(n=toy_r, p=toy_p)

(0, 100000)
(100000, 200000)
(200000, 300000)


2025-06-22 14:35:42,258 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 14:35:42,259 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 14:35:42,259 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 14:35:42,259 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 14:35:42,259 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 14:35:42,260 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/

We have narrowed it down to the range (200000, 300000)

In [ ]:
start=200000
interval=50000
for i in range(1,11):
    print((start+interval*(i-1),start+interval*i))
    toy_r=r[start+interval*(i-1):start+interval*i]
    toy_p=p[start+interval*(i-1):start+interval*i]
    np.random.negative_binomial(n=toy_r, p=toy_p)

(200000, 250000)
(250000, 300000)


2025-06-22 14:38:35,408 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 14:38:35,408 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
2025-06-22 14:38:35,409 - distributed.nanny - ERROR - Worker process died unexpectedly
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/distributed/process.py", line 190, in _run
    target(*args, **kwargs)
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/distributed/nanny.py", line 988, in _run
    asyncio.run(run())
  File "/opt/anaconda3/

Something in the range (250000, 300000)

In [ ]:
start=250000
interval=10000
for i in range(1,11):
    print((start+interval*(i-1),start+interval*i))
    toy_r=r[start+interval*(i-1):start+interval*i]
    toy_p=p[start+interval*(i-1):start+interval*i]
    np.random.negative_binomial(n=toy_r, p=toy_p)

(250000, 260000)


2025-06-22 14:43:16,417 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 14:43:16,417 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 14:43:16,417 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 14:43:16,417 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 14:43:16,418 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 14:43:16,418 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 14:43:16,418 - distributed.nanny - ERROR - Worker process died unexpectedly
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call

Something in the range (250000, 260000). Quick check, is it JUST something in that range?

In [ ]:
start=250000
interval=10000
for i in range(2,11):
    print((start+interval*(i-1),start+interval*i))
    toy_r=r[start+interval*(i-1):start+interval*i]
    toy_p=p[start+interval*(i-1):start+interval*i]
    np.random.negative_binomial(n=toy_r, p=toy_p)

(260000, 270000)


2025-06-22 14:52:02,732 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 14:52:02,732 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 14:52:02,733 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 14:52:02,733 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py"

No, it looks like the next section is borked as well. Let's inspect some values.

In [12]:
r[260000:270000]

array([0.00761527, 0.00761527, 0.00761527, ..., 0.07129764, 0.07129764,
       0.07129764], dtype=float32)

In [13]:
p[260000:270000]

array([7.37381939e-05, 7.37381939e-05, 7.37381939e-05, ...,
       1.56599487e-01, 1.56599487e-01, 1.56599487e-01])

In [15]:
np.random.negative_binomial(n=0.00761527, p=7.37381939e-05)
np.random.negative_binomial(n=0.07129764, p=1.56599487e-01)

0

Are any negative or zero?

In [20]:
any(r<0) or any(p<0) or any(r==0) or any(p==0)

False

Hrm, let's continue breaking it down.

In [ ]:
#(250000, 260000)
start=250000
interval=1000
for i in range(1,11):
    print((start+interval*(i-1),start+interval*i))
    toy_r=r[start+interval*(i-1):start+interval*i]
    toy_p=p[start+interval*(i-1):start+interval*i]
    np.random.negative_binomial(n=toy_r, p=toy_p)

(250000, 251000)


2025-06-22 15:27:51,582 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:27:51,582 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:27:51,582 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 15:27:51,582 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 15:27:51,583 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 15:27:51,583 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:27:51,583 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call

In [ ]:
#(250000, 251000)
start=250000
interval=100
for i in range(1,11):
    beg=start+interval*(i-1)
    end=start+interval*i
    print(beg,end)
    toy_r=r[beg:end]
    toy_p=p[beg:end]
    np.random.negative_binomial(n=toy_r, p=toy_p)

250000 250100
250100 250200
250200 250300
250300 250400
250400 250500
250500 250600
250600 250700


Now we're getting somewhere. (250600 250700)

In [ ]:
start=250600
interval=10
for i in range(1,11):
    beg=start+interval*(i-1)
    end=start+interval*i
    print(beg,end)
    toy_r=r[beg:end]
    toy_p=p[beg:end]
    np.random.negative_binomial(n=toy_r, p=toy_p)

250600 250610


2025-06-22 15:36:29,212 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:36:29,212 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:36:29,212 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 15:36:29,212 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 15:36:29,212 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 15:36:29,213 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/envs/env_tensorzinb/lib/pyth

250600 250610

In [11]:
r[250600:250610]

array([0.00761527, 0.00761527, 0.00761527, 0.00761527, 0.00761527,
       0.00761527, 0.00761527, 0.00761527, 0.00761527, 0.00761527],
      dtype=float32)

In [12]:
p[250600:250610]

array([4.58092312e-109, 4.58092312e-109, 4.58092312e-109, 4.58092312e-109,
       4.58092312e-109, 4.58092312e-109, 4.58092312e-109, 4.58092312e-109,
       4.58092312e-109, 4.58092312e-109])

I bet there's something about the magnitude of p that's throwing the routine off.

In [13]:
min(p)

5.247322732074934e-141

In [ ]:
np.random.negative_binomial(n=[0.00761527], p=[4.58092312e-109])

2025-06-22 15:46:56,561 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:46:56,561 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:46:56,561 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 15:46:56,561 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
2025-06-22 15:46:56,561 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-06-22 15:46:56,562 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-06-22 15:46:56,562 - distributed.nanny - ERROR - Worker process died unexpectedly
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/envs

In [8]:
batch.simulate_many(client,1)

[timing] extract arrays: 0.0001 seconds
[timing] sample ZINB: 0.0002 seconds
[timing] assign values: 0.0003 seconds
[timing] extract arrays: 0.0001 seconds
[timing] extract arrays: 0.0000 seconds
[timing] extract arrays: 0.0000 seconds
[timing] extract arrays: 0.0001 seconds
[timing] sample ZINB: 0.0084 seconds
[timing] assign values: 0.0002 seconds
[timing] sample ZINB: 0.0125 seconds
[timing] assign values: 0.0007 seconds
[timing] extract arrays: 0.0000 seconds
[timing] extract arrays: 0.0001 seconds
[timing] extract arrays: 0.0001 seconds
[timing] sample ZINB: 0.0041 seconds
[timing] assign values: 0.0002 seconds
[timing] extract arrays: 0.0001 seconds
[timing] sample ZINB: 0.0036 seconds
[timing] assign values: 0.0002 seconds


ERROR:tornado.application:Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOLoop object at 0x3290853f0>>, <Task finished name='Task-28266' coro=<ProfileTimePlot.trigger_update.<locals>.cb() done, defined at /opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/distributed/utils.py:739> exception=AttributeError("'NoneType' object has no attribute 'add_next_tick_callback'")>)
Traceback (most recent call last):
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/tornado/ioloop.py", line 782, in _discard_future_result
    future.result()
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/distributed/utils.py", line 741, in wrapper
    return await func(*args, **kwargs)
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-pa

[timing] extract arrays: 0.0001 seconds
[timing] extract arrays: 0.0000 seconds
[timing] sample ZINB: 0.0083 seconds
[timing] assign values: 0.0002 seconds
[timing] sample ZINB: 0.0096 seconds
[timing] assign values: 0.0002 seconds
[timing] extract arrays: 0.0000 seconds
[timing] extract arrays: 0.0000 seconds
[timing] extract arrays: 0.0001 seconds


KeyboardInterrupt: 

2025-06-20 16:27:54,691 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/distributed/process.py", line 190, in _run
    target(*args, **kwargs)
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/site-packages/distributed/nanny.py", line 988, in _run
    asyncio.run(run())
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/asyncio/runners.py", line 44, in run
    return loop.run_until_complete(main)
  File "/opt/anaconda3/envs/env_tensorzinb/lib/python3.10/asyncio/ba

In [ ]:
batch.fit_to_simulations(client)

In [ ]:
batch._flatten_all_parameters()

In [ ]:
batch.plot_nb_spread()

In [17]:
cluster.close()